# 05 Result Summary for Report


## Objective

汇总第二阶段新增的 CSV 与 PNG 产物，导出简化 LaTeX 表格，并生成 `stage2_experiment_design_fix_summary.md`。本 notebook 仍不写正式报告正文。


In [1]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = Path.cwd().resolve().parent if (Path.cwd().resolve().parent / "src").exists() else PROJECT_ROOT

sys.path.insert(0, str(PROJECT_ROOT / "src"))
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)
print(f"PROJECT_ROOT = {PROJECT_ROOT}")

from config import FIGURES_DIR, REPORT_TABLES_DIR, RESULTS_DIR
from data_utils import ensure_project_dirs

ensure_project_dirs()


PROJECT_ROOT = C:\Users\qintian\Desktop\大数据\Big-Data-Homework\期末考查报告_数字生活方式分析


## Load Key Result Tables


In [2]:
def read_result(filename):
    path = RESULTS_DIR / filename
    if not path.exists():
        raise FileNotFoundError(f"Missing result file: {path}. Run previous notebooks first.")
    return pd.read_csv(path)

classification_metrics = read_result("classification_tuned_metrics.csv")
classification_thresholds = read_result("classification_threshold_tuning.csv")
regression_productivity = read_result("regression_productivity_metrics.csv")
regression_dependence = read_result("regression_digital_dependence_metrics.csv")
regression_comparison = read_result("regression_target_comparison.csv")
clustering_comparison = read_result("clustering_model_comparison.csv")
clustering_profiles = read_result("clustering_lifestyle_profiles.csv")
clustering_profiles_compact = read_result("clustering_lifestyle_profiles_compact.csv")

display(classification_metrics)
display(regression_comparison)
display(clustering_comparison.sort_values("silhouette", ascending=False).head(10))
display(clustering_profiles_compact)


,dataset,model,threshold_policy,threshold,best_params,accuracy,precision,recall,f1,balanced_accuracy,roc_auc,pr_auc,tn,fp,fn,tp
0,validation,gradient_boosting,default_0_50,0.50,"{'model__n_estimators': 100, 'model__max_depth...",0.817352,0.642857,0.204545,0.310345,0.587987,0.723283,0.480507,510,15,105,27
1,validation,random_forest,default_0_50,0.50,"{'model__n_estimators': 400, 'model__min_sampl...",0.808219,0.535714,0.340909,0.416667,0.633312,0.709798,0.475859,486,39,87,45
2,validation,logistic_regression,default_0_50,0.50,"{'model__C': 0.01, 'model__class_weight': None}",0.811263,0.642857,0.136364,0.225000,0.558658,0.720087,0.454345,515,10,114,18
3,test,gradient_boosting,default_0_50,0.50,"{'model__n_estimators': 100, 'model__max_depth...",0.827429,0.681159,0.267045,0.383673,0.617786,0.753081,0.508394,677,22,129,47
4,test,gradient_boosting,max_f1,0.35,"{'model__n_estimators': 100, 'model__max_depth...",0.829714,0.595745,0.477273,0.529968,0.697864,0.753081,0.508394,642,57,92,84
5,test,gradient_boosting,recall_at_least_60_best_precision,0.14,"{'model__n_estimators': 100, 'model__max_depth...",0.776000,0.459350,0.642045,0.535545,0.725887,0.753081,0.508394,566,133,63,113
6,test,gradient_boosting,recall_at_least_70_best_precision,0.12,"{'model__n_estimators': 100, 'model__max_depth...",0.488000,0.253623,0.795455,0.384615,0.603021,0.753081,0.508394,287,412,36,140


,model,target,cv_best_r2,cv_best_mse,cv_best_mae,n_train,n_test,feature_count,best_params,mae,mse,rmse,r2
0,gradient_boosting,productivity_score,0.011926,94.955100,7.614592,2625,875,22,"{'model__n_estimators': 100, 'model__max_depth...",7.167096,85.203131,9.230554,-0.004064
1,gradient_boosting,digital_dependence_score,0.980448,3.897547,1.126590,2625,875,22,"{'model__n_estimators': 200, 'model__max_depth...",0.998244,3.147071,1.773999,0.983901


,algorithm,k,inertia,silhouette,calinski_harabasz,davies_bouldin
15,kmeans,3,38902.059054,0.185960,611.177026,1.795865
14,kmeans,2,43202.432147,0.180775,752.802348,1.906514
18,kmeans,6,30676.959005,0.155473,497.113845,1.644684
19,kmeans,7,28584.283381,0.152058,487.083508,1.541899
1,agglomerative,3,NaN,0.151769,495.893794,1.876770
2,agglomerative,4,NaN,0.149575,429.188941,1.812652
7,gaussian_mixture,2,NaN,0.147831,650.846092,2.159854
17,kmeans,5,33054.139274,0.143342,514.032713,1.780577
16,kmeans,4,35740.788027,0.140636,546.436423,1.961482
20,kmeans,8,27317.685419,0.136497,459.862441,1.588970


,cluster,cluster_algorithm,cluster_size,device_hours_per_day,social_media_mins,sleep_hours,sleep_quality,high_risk_flag,productivity_score,digital_dependence_score,suggested_cluster_label
0,0,kmeans,447,6.953311,430.212528,7.381558,2.789936,0.196868,65.337548,34.329610,high_social_media_profile
1,1,kmeans,1039,11.053782,131.878730,6.121790,1.695291,0.372474,65.908133,51.242518,high_device_dependence_profile
2,2,kmeans,2014,5.471132,113.427507,7.810619,3.213733,0.114201,64.976723,29.696236,balanced_low_load_profile


## Export LaTeX Table Snippets

正文表格只保留核心字段；完整画像表保留给附录或结果 CSV。


In [3]:
def export_latex_table(df, filename, caption, label, columns=None, max_rows=None):
    table_df = df.copy()
    if columns is not None:
        table_df = table_df[[column for column in columns if column in table_df.columns]]
    if max_rows is not None:
        table_df = table_df.head(max_rows)
    text = table_df.to_latex(
        index=False,
        escape=True,
        float_format=lambda value: f"{value:.4f}",
        caption=caption,
        label=label,
    )
    path = REPORT_TABLES_DIR / filename
    path.write_text(text, encoding="utf-8")
    return path

classification_test = classification_metrics[classification_metrics["dataset"] == "test"].copy()
clustering_best = (
    clustering_comparison.dropna(subset=["silhouette"])
    .sort_values(["algorithm", "silhouette"], ascending=[True, False])
    .groupby("algorithm")
    .head(1)
)

paths = [
    export_latex_table(
        classification_test,
        "classification_tuned_metrics.tex",
        "Tuned classification test metrics under threshold policies",
        "tab:classification-tuned-metrics",
        columns=["model", "threshold_policy", "threshold", "precision", "recall", "f1", "roc_auc", "pr_auc", "balanced_accuracy"],
    ),
    export_latex_table(
        classification_thresholds,
        "classification_threshold_tuning.tex",
        "Validation threshold tuning results",
        "tab:classification-threshold-tuning",
        columns=["model", "policy", "threshold", "precision", "recall", "f1", "pr_auc", "balanced_accuracy"],
    ),
    export_latex_table(
        regression_productivity,
        "regression_productivity_metrics.tex",
        "Productivity score regression metrics",
        "tab:regression-productivity-metrics",
        columns=["model", "cv_best_r2", "mae", "mse", "rmse", "r2"],
    ),
    export_latex_table(
        regression_dependence,
        "regression_digital_dependence_metrics.tex",
        "Digital dependence score regression metrics",
        "tab:regression-digital-dependence-metrics",
        columns=["model", "cv_best_r2", "mae", "mse", "rmse", "r2"],
    ),
    export_latex_table(
        regression_comparison,
        "regression_target_comparison.tex",
        "Best regression result by target",
        "tab:regression-target-comparison",
        columns=["target", "model", "mae", "mse", "rmse", "r2", "cv_best_r2"],
    ),
    export_latex_table(
        clustering_best,
        "clustering_model_comparison.tex",
        "Best clustering result by algorithm",
        "tab:clustering-model-comparison",
        columns=["algorithm", "k", "inertia", "silhouette", "calinski_harabasz", "davies_bouldin"],
    ),
    export_latex_table(
        clustering_profiles_compact,
        "clustering_lifestyle_profiles_compact.tex",
        "Compact lifestyle cluster profiles",
        "tab:clustering-profiles-compact",
        columns=[
            "cluster",
            "cluster_size",
            "device_hours_per_day",
            "social_media_mins",
            "sleep_hours",
            "sleep_quality",
            "high_risk_flag",
            "productivity_score",
            "digital_dependence_score",
            "suggested_cluster_label",
        ],
    ),
]
for path in paths:
    print(path)


C:\Users\qintian\Desktop\大数据\Big-Data-Homework\期末考查报告_数字生活方式分析\report\tables\classification_tuned_metrics.tex
C:\Users\qintian\Desktop\大数据\Big-Data-Homework\期末考查报告_数字生活方式分析\report\tables\classification_threshold_tuning.tex
C:\Users\qintian\Desktop\大数据\Big-Data-Homework\期末考查报告_数字生活方式分析\report\tables\regression_productivity_metrics.tex
C:\Users\qintian\Desktop\大数据\Big-Data-Homework\期末考查报告_数字生活方式分析\report\tables\regression_digital_dependence_metrics.tex
C:\Users\qintian\Desktop\大数据\Big-Data-Homework\期末考查报告_数字生活方式分析\report\tables\regression_target_comparison.tex
C:\Users\qintian\Desktop\大数据\Big-Data-Homework\期末考查报告_数字生活方式分析\report\tables\clustering_model_comparison.tex
C:\Users\qintian\Desktop\大数据\Big-Data-Homework\期末考查报告_数字生活方式分析\report\tables\clustering_lifestyle_profiles_compact.tex


## Artifact Manifest and Stage 2 Summary


In [4]:
result_files = sorted(RESULTS_DIR.glob("*.csv")) + sorted(RESULTS_DIR.glob("*.txt")) + sorted(RESULTS_DIR.glob("*.md"))
figure_files = sorted(FIGURES_DIR.glob("*.png"))
manifest = pd.DataFrame(
    [{"type": "result", "path": str(path.relative_to(PROJECT_ROOT))} for path in result_files]
    + [{"type": "figure_png", "path": str(path.relative_to(PROJECT_ROOT))} for path in figure_files]
)
manifest.to_csv(RESULTS_DIR / "report_artifact_manifest.csv", index=False)

best_classification = classification_test.sort_values(["f1", "recall"], ascending=False).iloc[0]
best_prod = regression_productivity.sort_values(["cv_best_r2", "r2"], ascending=False).iloc[0]
best_dep = regression_dependence.sort_values(["cv_best_r2", "r2"], ascending=False).iloc[0]
best_cluster = clustering_comparison.dropna(subset=["silhouette"]).sort_values("silhouette", ascending=False).iloc[0]
prod_weak = best_prod["r2"] <= 0.05
dep_better = best_dep["r2"] > best_prod["r2"]
cluster_weak = best_cluster["silhouette"] < 0.20

summary = f"""# Stage 2 Experiment Design Fix Summary

## 1. 修复内容

本阶段补充了分类调参、分类阈值选择、双目标回归、聚类算法对比、聚类特征范围控制、EDA 图表和报告表格简化。所有新增结果均由 notebook 真实运行生成。

## 2. 分类调参和阈值调优结果

分类任务继续预测 `high_risk_flag`，并严格排除心理状态、效率、数字依赖、ID 和目标列。调参结果保存为 `results/classification_cv_results.csv`，阈值调优结果保存为 `results/classification_threshold_tuning.csv`。

测试集上当前综合 F1 较高的策略为 `{best_classification['threshold_policy']}`，模型为 `{best_classification['model']}`，threshold={best_classification['threshold']:.2f}，Precision={best_classification['precision']:.4f}，Recall={best_classification['recall']:.4f}，F1={best_classification['f1']:.4f}，PR-AUC={best_classification['pr_auc']:.4f}，ROC-AUC={best_classification['roc_auc']:.4f}。

## 3. productivity_score 是否仍然弱预测

productivity_score 当前最佳模型为 `{best_prod['model']}`，R2={best_prod['r2']:.4f}，MSE={best_prod['mse']:.4f}，RMSE={best_prod['rmse']:.4f}，MAE={best_prod['mae']:.4f}。

{'该目标在当前特征下仍然属于弱预测或负结果，不能写成数字行为可以有效预测生产力。' if prod_weak else '该目标已出现一定可预测性，但仍需在正式报告中谨慎解释。'}

## 4. digital_dependence_score 是否更适合作为回归主线

digital_dependence_score 当前最佳模型为 `{best_dep['model']}`，R2={best_dep['r2']:.4f}，MSE={best_dep['mse']:.4f}，RMSE={best_dep['rmse']:.4f}，MAE={best_dep['mae']:.4f}。

{'digital_dependence_score 的效果优于 productivity_score，更适合作为回归主线；productivity_score 可作为辅助负结果分析。' if dep_better else 'digital_dependence_score 未明显优于 productivity_score，回归结论仍需谨慎。'}

## 5. 聚类特征范围

聚类已经改成只使用数字行为和生活习惯数值特征，不再把 gender、region、income_level、education_level、daily_role、device_type 等背景类别变量放入聚类训练。背景类别和结果变量只用于聚类后画像解释。

## 6. 聚类算法对比结果

聚类比较结果保存为 `results/clustering_model_comparison.csv`。当前 Silhouette 最高的模型为 `{best_cluster['algorithm']}`，k={int(best_cluster['k'])}，Silhouette={best_cluster['silhouette']:.4f}，Calinski-Harabasz={best_cluster['calinski_harabasz']:.4f}，Davies-Bouldin={best_cluster['davies_bouldin']:.4f}。

{'轮廓系数仍然偏低，聚类只能作为探索性画像，不能作为严格人群边界。' if cluster_weak else '轮廓系数已有一定可用性，但仍应将聚类解释限定为探索性画像。'}

## 7. 可以进入正式报告的结果

- 数据集合规性、字段结构、样本规模和合成数据限制。
- 分类任务的泄漏控制、调参流程、PR-AUC/ROC-AUC、阈值调优和混淆矩阵。
- digital_dependence_score 回归结果，如果其 R2 明显优于 productivity_score。
- 聚类输入特征白名单、算法对比和简化画像表。

## 8. 需要谨慎解释的结果

- productivity_score 若 R2 接近 0 或为负，只能作为弱预测/负结果分析。
- 聚类 Silhouette 若低于 0.20，只能写探索性用户画像，不写严格分群边界。
- 所有 EDA 相关性图只支持描述性分析，不支持因果结论。
"""

summary_path = RESULTS_DIR / "stage2_experiment_design_fix_summary.md"
summary_path.write_text(summary, encoding="utf-8")
display(manifest)
print(summary_path)


,type,path
0,result,results\classification_best_confusion_matrix.csv
1,result,results\classification_cv_results.csv
2,result,results\classification_feature_exclusion.csv
3,result,results\classification_final_confusion_matrix.csv
4,result,results\classification_high_risk_metrics.csv
...,...,...
60,figure_png,figures\regression_digital_dependence_residual...
61,figure_png,figures\regression_productivity_observed_vs_pr...
62,figure_png,figures\regression_productivity_permutation_im...
63,figure_png,figures\regression_productivity_residuals.png


C:\Users\qintian\Desktop\大数据\Big-Data-Homework\期末考查报告_数字生活方式分析\results\stage2_experiment_design_fix_summary.md
